# 05 — Quantum Kernel SVM (QSVM)
A small-sample proof-of-concept using PennyLane's CPU simulator. Quantum kernel evaluation is expensive (quadratic in sample count), so start with a stratified training subset. Keep the same held-out test split for final comparisons and clearly report sample counts.

In [ ]:
%pip -q install pennylane scikit-learn

In [ ]:
import pennylane as qml
print("PennyLane:",qml.__version__)

In [ ]:
from pathlib import Path
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif, SelectKBest, RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve)
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
# If running from a notebook in /notebooks, move to project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "dataset" / "diabetes.csv"
if not DATA_PATH.exists():
    # Colab fallback: upload diabetes.csv into the current session or mount Drive.
    DATA_PATH = Path("/content/diabetes.csv")
df = pd.read_csv(DATA_PATH)
TARGET = "Outcome"
FEATURES = [c for c in df.columns if c != TARGET]
INVALID_ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils import resample
X=df.drop(columns=TARGET).replace({c:{0:np.nan} for c in INVALID_ZERO_COLS})
y=df[TARGET].astype(int)
Xtr_raw,Xte_raw,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
# Fit preprocessing and MI selection on training only
imp=KNNImputer(n_neighbors=5,weights="distance")
Xtr=imp.fit_transform(Xtr_raw); Xte=imp.transform(Xte_raw)
sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
selector=SelectKBest(mutual_info_classif,k=4)
Xtr=selector.fit_transform(Xtr,y_train); Xte=selector.transform(Xte)
# Quantum angle embedding expects bounded inputs
mm=MinMaxScaler(feature_range=(-np.pi,np.pi))
Xtr=mm.fit_transform(Xtr); Xte=mm.transform(Xte)
# Stratified subset for kernel training
if len(Xtr) > 250:
    _, Xq, _, yq = train_test_split(
        Xtr, np.asarray(y_train), test_size=250, stratify=np.asarray(y_train),
        random_state=42
    )
else:
    Xq, yq = Xtr, np.asarray(y_train)
yq=np.asarray(yq)
n_qubits=4
dev=qml.device("default.qubit",wires=n_qubits)
def feature_map(x):
    qml.AngleEmbedding(x,wires=range(n_qubits),rotation="Y")
    for i in range(n_qubits-1):
        qml.CNOT(wires=[i,i+1])
    qml.CNOT(wires=[n_qubits-1,0])
@qml.qnode(dev)
def kernel_circuit(x1,x2):
    feature_map(x1)
    qml.adjoint(feature_map)(x2)
    return qml.probs(wires=range(n_qubits))
def quantum_kernel(A,B):
    K=np.zeros((len(A),len(B)))
    for i,a in enumerate(A):
        for j,b in enumerate(B):
            K[i,j]=kernel_circuit(a,b)[0]
    return K
K_train=quantum_kernel(Xq,Xq)
K_test=quantum_kernel(Xte,Xq)
qsvm=SVC(kernel="precomputed",class_weight="balanced")
qsvm.fit(K_train,yq)
pred=qsvm.predict(K_test)
scores=qsvm.decision_function(K_test)
print(classification_report(y_test,pred,digits=3))
print("ROC-AUC (decision scores):",roc_auc_score(y_test,scores))

If kernel evaluation is too slow, reduce the training subset to 100–150 samples for a smoke test. Report this limitation honestly. The kernel is computed from state overlap probabilities; no quantum speedup is claimed.